# Part 1: multimodal, conversation-aware RAG

This is the revised implementation, not the earlier baseline. Run it in a fresh GPU kernel,
one notebook at a time. A nominal 16-GB GPU is the lab target. The RTX 4050 uses a CPU-offloaded
low-VRAM profile, which must pass the real preflight on your own PC. A cold setup needs internet,
roughly 40 GB free disk overall and enough host RAM for offload. No hosted model API is used.

The default is a full run. `LAB2_SMOKE_ONLY=1` explicitly skips model-backed measurements;
it is a developer check, never a completed experiment. Real human ratings are never generated.
Device-memory sampling covers the actual subprocess stages but is not proof that an RTX 4090
is a 16-GB GPU. Keep the printed hardware identity and all outputs.

Before leaving the computer, run through the real GPU preflight and confirm it passes.
Matching completed items resume from atomic checkpoints. Changed code, data or hardware starts
a separate checkpoint namespace; no old result is relabeled. Keep the machine awake and the
local GPU runtime running. Closing the local server or rebooting stops execution.


In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)

if not (ROOT / "tools/response_schemas.py").is_file():
    old_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-structured-v1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/response_schemas.py").is_file():
        branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        if branch == "main" and not dirty:
            subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    if not (ROOT / "tools/response_schemas.py").is_file():
        raise RuntimeError(f"The safe sibling checkout at {ROOT} is outdated. Your original files at {old_root} were preserved.")
    print("Using schema-constrained code at", ROOT)
    print("The earlier checkout, local edits and results are unchanged at", old_root)

subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")

# Permit first-time downloads during preflight, even after restarting this notebook.
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
# Monitor actual device memory in each subprocess. A larger GPU is not relabeled as a T4.
_run_stage = run_script
def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    _run_stage("tools/monitored_run.py", "--record", record, "--", PYTHON, ROOT / relative, *arguments)


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "4488ba5",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Check the implementation
These offline regression tests exercise real PDF extraction, Qdrant storage with temporary vectors, exact figure identity and isolated conversation state. Mock model outputs are test fixtures only.

In [2]:
run_script("tools/test_lab_completion.py")
run_script("tools/test_structured_responses.py")
run_script("tools/test_constrained_decoding.py")
run_script("tools/test_overnight_resume.py")

Running monitored_run.py


test_actual_tiny_qwen_multimodal_forward (__main__.TestAgentSafety.test_actual_tiny_qwen_multimodal_forward) ... `Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


ok


test_image_instructions_are_data_and_refused (__main__.TestAgentSafety.test_image_instructions_are_data_and_refused) ... ok


test_instruction_normalization (__main__.TestAgentSafety.test_instruction_normalization) ... ok


test_json_decode_fails_closed (__main__.TestAgentSafety.test_json_decode_fails_closed) ... ok


... (83 outputs left out)

----------------------------------------------------------------------


Ran 13 tests in 4.518s


OK


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/test_overnight_resume_1794277296395304000.memory.json


## Locate the read-only data
On Kaggle attach the supplied competition data. `QUESTIONS_PATH` supports the hidden set without code edits. No question-specific rules or answer lookup is used.

In [3]:
P1 = ROOT / "part1_rag"
DATA = Path(os.environ.get("LAB2_DATA_DIR", str(P1 / "data"))).resolve()
if Path("/kaggle/input").is_dir() and "LAB2_DATA_DIR" not in os.environ:
    manifests = list(Path("/kaggle/input").rglob("kb_manifest.json"))
    if manifests:
        DATA = manifests[0].parent
QUESTIONS = Path(os.environ.get("QUESTIONS_PATH", str(DATA / "Questions.csv"))).resolve()
OUTPUT = P1 / "outputs/v2/submission.csv"
if not QUESTIONS.is_file():
    raise FileNotFoundError("Attach Questions.csv or set QUESTIONS_PATH to the supplied hidden-set file.")
print("Data:", DATA, "Questions:", QUESTIONS)

Data: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part1_rag/data Questions: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part1_rag/data/Questions.csv


## Real GPU preflight before leaving it overnight
This loads the actual embedding, image and VLM paths on this GPU and checks a grounded chart response. It also fills the public model cache before long stages start. Do not leave an unpassed preflight unattended. This is not the final answer-quality evaluation.

In [4]:
if SMOKE_ONLY:
    print("SKIPPED: real GPU preflight; no GPU readiness is claimed.")
else:
    run_script("tools/overnight_preflight.py", "--part", "part1_rag", "--data-dir", DATA)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

SKIPPED: real GPU preflight; no GPU readiness is claimed.


## Ingest, retrieve and generate
Numbered figures/tables retain exact labels, including the corrected version. Trusted text and CLIP image vectors are stored in local Qdrant. Appendix C is quarantined, never used as answer evidence. Each conversation starts with independent memory; user assumptions never modify the source PDFs or database. Decoding is constrained to the required JSON fields, valid status values and the actual retrieved source labels. This prevents format hallucinations, not factual mistakes. Persistent failures remain visible and retain a diagnostic; answers are never filled from a lookup table.

In [5]:
if SMOKE_ONLY:
    print("SKIPPED: full VLM answers and competition submission. Unit-test vectors are not evaluation embeddings.")
else:
    run_script("part1_rag/src/generate_submission.py", "--data-dir", DATA,
               "--questions-path", QUESTIONS, "--output", OUTPUT, "--require-dense")
    run_script("part1_rag/data/validate_submission.py", OUTPUT, QUESTIONS)
    import csv
    with OUTPUT.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(json.dumps(rows[:3], indent=2))
    print("Validated submission:", OUTPUT)

SKIPPED: full VLM answers and competition submission. Unit-test vectors are not evaluation embeddings.


## Dense-only baseline versus the final system
The 25 candidate golden references are AI-assisted proposals committed before this run. Both team members must review them and replace or correct them as needed. Their `team_verified` flags remain false until that happens. Numeric/status checks and VLM citation entailment are labeled automatic proxies, not fabricated human results.

In [6]:
if SMOKE_ONLY:
    print("SKIPPED: golden-set model evaluation and human ratings.")
else:
    run_script("part1_rag/src/evaluate_rag.py", "--data-dir", DATA,
               "--output", P1 / "outputs/v2/evaluation",
               "--golden", os.environ.get("LAB2_GOLDEN_PATH", str(P1 / "eval/golden_candidate.json")))
    print((P1 / "outputs/v2/evaluation/evaluation.json").read_text())

SKIPPED: golden-set model evaluation and human ratings.


## Review and export
Complete the real groundedness form for both systems, refresh the report, and save this executed notebook. Upload/select the validated submission on Kaggle manually; the code does not claim a leaderboard submission.

In [7]:
run_script("tools/prepare_rating_forms.py")
evaluation_folder = P1 / "outputs/v2/evaluation"
if not SMOKE_ONLY and (evaluation_folder / "evaluation.json").is_file():
    run_script("part1_rag/src/evaluate_rag.py", "--refresh-review", "--output", evaluation_folder,
               "--golden", os.environ.get("LAB2_GOLDEN_PATH", str(P1 / "eval/golden_candidate.json")))
run_script("tools/final_submission_audit.py")
if not SMOKE_ONLY:
    run_script("tools/build_combined_report.py")
else:
    print("SKIPPED: rebuilding the submission report from a developer smoke run.")
run_script("tools/export_lab_results.py", "--part", "part1_rag")

Running monitored_run.py


Prepared blank forms: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/docs/rating_forms


No observed score or human judgment has been generated. The synthetic example is not an input rating file.


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/prepare_rating_forms_1794277301227953000.memory.json


Running monitored_run.py


Part 2 training comparison


... (36 outputs left out)

Result ZIP created on the runtime filesystem: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/part1_rag_20261110T022141611746Z.zip


Size MiB: 18.36


Save the executed notebook first and rerun this export cell to include its latest outputs. A local/Docker Colab runtime stores this on the GPU machine, not automatically in browser Downloads.


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/export_lab_results_1794277301547971000.memory.json


## Download one ZIP with all available lab results

Run this cell after the overnight work. It bundles all available results for Parts 1, 2 and 3, executed notebooks, source/configuration, reports, adapters, generated images and diagnostic logs. Base-model downloads, environments, original datasets and old archives are excluded. Missing files are listed honestly in `BUNDLE_MANIFEST.json`.

The ZIP transfers through the Colab notebook connection, including a local kernel, without a localhost hyperlink. A native Windows/Mac runtime also copies it into its accessible Downloads folder. Browser location follows your download settings. Keep the kernel connected until transfer finishes. If the run fails, you can execute only this export cell to send the partial evidence; no GPU work is repeated.

In [8]:
from pathlib import Path
import os, sys, subprocess, json, importlib.util

if "ROOT" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents,
        Path(os.environ.get("LAB2_ROOT", str(Path.home() / "DATA266/Lab-2"))),
        Path("/content/Lab-2"), Path("/kaggle/working/Lab-2")]
    ROOT = next((p.resolve() for p in candidates if (p / "tools/export_lab_results.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Set LAB2_ROOT to your existing Lab-2 folder. Do not rerun training just to export files.")

# This script only collects saved files. It does not install packages, train, or generate.
subprocess.run([sys.executable, str(ROOT / "tools/export_lab_results.py"), "--all"], cwd=ROOT, check=True)
bundle = json.loads((ROOT / "output/exports/latest_all_results.json").read_text())
print("Upload this ZIP in our chat:", bundle["archive"])
print("ZIP SHA256:", bundle["sha256"])
if globals().get("SMOKE_ONLY", False):
    print("CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.")
else:
    spec = importlib.util.spec_from_file_location("lab2_download_results", ROOT / "tools/download_results.py")
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloader.offer_download(bundle["archive"], auto_start=True)

ALL RESULTS ZIP: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/Lab2_All_Results_20261110T022142437549Z.zip
Size MiB: 142.21 Files: 446
SHA256: 80d4d9c2cb2477bd96894d52c0f83b532f14891542809ce7747fc81920fa7646
Missing on THIS runtime (see manifest): part1_rag/outputs/v2/evaluation/evaluation.json, part3_agents/outputs/v2/latest_run.json, CONTRIBUTIONS.md, links.txt
No training or generation was repeated. Review the archive before sharing; common token patterns were redacted.
Upload this ZIP in our chat: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/Lab2_All_Results_20261110T022142437549Z.zip
ZIP SHA256: 80d4d9c2cb2477bd96894d52c0f83b532f14891542809ce7747fc81920fa7646
CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.
